In [1]:
from meshparty import *
import orjson
import numpy as np
import pandas as pd
from caveclient import CAVEclient

### Download data

In [2]:
# client = CAVEclient('minnie65_phase3_v1')
# tbl_qry = client.materialize.tables

# cell_df = tbl_qry.allen_column_mtypes_v2(cell_type="PTC").query(
#     desired_resolution=[1,1,1]
# )
# root_id = cell_df.iloc[10].pt_root_id
# with open("root_id.json", "w") as f:
#     f.write(
#         str(root_id)
#     )
# pre_syn_df = tbl_qry.synapses_pni_2(pre_pt_root_id=root_id).query(desired_resolution=[1,1,1], split_positions=True)
# post_syn_df = tbl_qry.synapses_pni_2(post_pt_root_id=root_id).query(desired_resolution=[1,1,1], split_positions=True)

# pre_syn_df.to_feather('pre.feather')
# post_syn_df.to_feather('post.feather')

# skel = client.skeleton.get_skeleton(root_id)
# with open('skel.json', 'bw') as f:
#     f.write(
#         orjson.dumps(skel, option=orjson.OPT_SERIALIZE_NUMPY)
#     )
# l2ids = skel['lvl2_ids']
# l2_df = client.l2cache.get_l2data_table(l2ids)
# l2_df.to_feather('l2properties.feather')
# l2_graph = client.chunkedgraph.level2_chunk_graph(root_id)
# with open("l2graph.json", "wb") as f:
#     f.write(
#         orjson.dumps(l2_graph, option=orjson.OPT_SERIALIZE_NUMPY)
#     )

### Load data locally

In [3]:
with open('root_id.json', 'r') as f:
    root_id = int(f.read())

pre_syn_df = pd.read_feather(
    'pre.feather'
)
post_syn_df = pd.read_feather(
    'post.feather',
)

with open('skel.json') as f:
    skel = orjson.loads(f.read())

with open('l2graph.json') as f:
    l2_graph = orjson.loads(f.read())

l2_df = pd.read_feather('l2properties.feather')
l2_df.reset_index(inplace=True)

In [4]:
import morphsync as sync

In [5]:
import fastremap
spatial_columns = ['ctr_pt_position_x', 'ctr_pt_position_y', 'ctr_pt_position_z']

properties = [x for x in l2_df.reset_index().columns.values if x not in spatial_columns]
l2_map = {v:k for k,v in l2_df['l2_id'].to_dict().items()}
edges = fastremap.remap(
    l2_graph,
    l2_map,
)
l2_spatial_columns = ['rep_coord_nm_x', 'rep_coord_nm_y', 'rep_coord_nm_z',]

In [6]:
l2_df_reidx = l2_df.set_index('l2_id')

In [7]:
cell = sync.MorphSync()
cell.add_graph(
    graph=(l2_df_reidx, edges),
    name='graph',
    spatial_columns=l2_spatial_columns,
)
cell.add_graph(
    graph=(np.array(skel['vertices']), np.array(skel['edges'])),
    name='skeleton',
)
cell.add_link(
    source='graph',
    target='skeleton',
    mapping=np.array(skel['mesh_to_skel_map']),
)
new_cell = cell.apply_mask(
    'skeleton',
    mask=np.array(skel['compartment'])==2,
)
new_cell._layers

{'skeleton': Graph(nodes=(14153, 3), edges=(14152, 2)),
 'graph': Graph(nodes=(21030, 27), edges=(0, 2))}

In [8]:
GraphSync(
    name='l2_graph',
    vertices=l2_df,
    spatial_columns=l2_spatial_columns,
    edges=edges,
    vertex_index='l2_id',
)

GraphSync(name=l2_graph, vertices=25889, edges=30352)

In [9]:
from caveclient import CAVEclient
client = CAVEclient('minnie65_phase3_v1')
ts = client.chunkedgraph.get_root_timestamps(root_id, latest=True)[0]
l2_ids = client.chunkedgraph.get_roots(pre_syn_df['pre_pt_supervoxel_id'], stop_layer=2, timestamp=ts)
pre_syn_df['pre_pt_l2_id'] = l2_ids

l2_ids = client.chunkedgraph.get_roots(post_syn_df['post_pt_supervoxel_id'], stop_layer=2, timestamp=ts)
post_syn_df['post_pt_l2_id'] = l2_ids

In [10]:
nrn = (
    MeshWorkSync(
        name=root_id,
    ).add_graph(
        vertices=l2_df,
        spatial_columns=l2_spatial_columns,
        edges=edges,
        vertex_index="l2_id",
    ).add_skeleton(
        vertices=np.array(skel['vertices']),
        edges=np.array(skel['edges']),
        labels={'radius': skel['radius'], 'compartment': skel['compartment']},
        linkage=Link(mapping=skel['mesh_to_skel_map'], source='graph', map_value_is_index=False)
    ).add_point_annotations(
        'pre_syn',
        vertices=pre_syn_df,
        spatial_columns='ctr_pt',
        vertex_index='id',
        linkage=Link(mapping='pre_pt_l2_id', target='graph')
    ).add_point_annotations(
        'post_syn',
        vertices=post_syn_df,
        spatial_columns='ctr_pt',
        vertex_index='id',
        linkage=Link(mapping='post_pt_l2_id', target='graph')
    )
)

Notes on the mapping functions:
* `get_mapping` tries to find a 1:1 mapping from source to target using links. It cannot handle a duplicate index on the target, but could you drop duplicates on the mapping_df target first? This would ensure that the map went to "a" target value even in the instance there is not THE target value.
* 

In [28]:
nrn.graph.map_index_to_layer('skeleton', source_index=[0,1,2,3,4], positional=True)

array([2, 0, 1, 2, 3])

In [38]:
nrn._morphsync.get_masking(
    'graph',
    'skeleton',
    source_index=nrn.graph.vertex_index[0:10],
)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [35]:
nrn.skeleton.layer.name

AttributeError: 'Graph' object has no attribute 'name'

In [14]:
3000 * 0.175

525.0

In [15]:
3000 * 0.075

225.0

In [ ]:
np.unique(nrn.graph.vertex_index).shape

In [ ]:
nrn.skeleton.vertex_index

In [ ]:
import pdb; pdb.pm()

In [ ]:
nrn.skeleton.layer.facets[0]

In [ ]:
skel['root']

In [ ]:
from scipy import sparse

In [ ]:
nrn.csgraph.

In [ ]:
skel['root']

In [ ]:
nrn.skeleton.csgraph.T.sum(axi)

In [ ]:
np.flatnonzero(nrn.skeleton.csgraph_binary.sum(axis=1) == 0)

In [ ]:
sparse.csgraph.dijkstra(
    nrn.skeleton.csgraph.T,
    indices=skel['root'],
)

In [ ]:
nrn.skeleton.edges

In [ ]:
nrn.graph.edges_positional

In [ ]:
nrn.annotations.pre_syn.map_index_to_layer('skeleton', source_index=[0,1,2], positional=True)

In [ ]:
(nrn.annotations.pre_syn.nodes['size'] > 2000).values

In [ ]:
nrn.annotations.pre_syn.map_mask_to_layer('skeleton', (nrn.annotations.pre_syn.nodes['size'] > 2000).values)

In [ ]:
nrn.skeleton.map_index_to_layer('graph', source_index=[0])

In [ ]:
nrn._morphsync.

In [ ]:
fastremap.remap(
    nrn._morphsync.get_mapping('pre_syn', 'graph', source_index=nrn.annotations.pre_syn.vertex_index),
    {int(k):ii for ii, k in enumerate(np.array(nrn._morphsync._layers['graph'].vertices_index.values))},
)

In [ ]:
nrn._morphsync.get_mapping('pre_syn', 'graph', source_index=nrn.annotations.pre_syn.vertex_index)

In [ ]:
nrn.annotations.pre_syn.vertex_index

In [ ]:
nrn.annotations.pre_syn.nodes

In [ ]:
nrn._morphsync.get_mapping('pre_syn', 'skeleton', source_index=nrn.layers['pre_syn'].nodes.index)

In [ ]:
import polyscope as ps

In [ ]:
ps.init()

In [ ]:
ps_nrn = ps.register_curve_network(
    str(nrn.name),
    nodes=np.array(nrn.skeleton.vertices),
    edges=np.array(nrn.skeleton.edges),
)
ps_nrn.add_scalar_quantity('radius', nrn.skeleton.nodes['radius'].values)
ps_nrn.set_node_radius_quantity('radius', autoscale=False)
ps_nrn.add_scalar_quantity('compartment', nrn.skeleton.nodes['compartment'].values)

In [ ]:
ps.show()

In [ ]:
nrn.layer_df

In [ ]:
nrn.skeleton.add_label('radius', skel['radius'])

In [ ]:
ps.show()

In [ ]:
nrn.graph.edges_positional

In [ ]:
nrn._morphsync.apply_mask(
    'skeleton',
    mask=np.array(skel['compartment'])==1,
).layers

In [ ]:
nrn._morphsync.apply_mask(
    'skeleton',
    mask=np.array(skel['compartment'])==2,
).layers

In [ ]:
nrn._morphsync.add_link(
    source='graph',
    target='skeleton',
    mapping=np.array(skel['mesh_to_skel_map']),
)


In [ ]:
nrn.skeleton.add_label(
    skel['compartment'],
    name='compartment'
)

In [ ]:
new_morphsync = nrn.skeleton._morphsync.apply_mask(
    layer_name='skeleton',
    mask=(nrn.skeleton.labels['compartment']==2).values
)

In [ ]:
nrn.graph

In [ ]:
new_morphsync.graph

In [ ]:
nrn.skeleton.apply_mask(
    mask=(nrn.skeleton.labels['compartment']==2).values
)

In [ ]:
nrn._morphsync.layers

In [ ]:
nrn._morphsync.add_link(source=)

In [ ]:
nrn.

In [ ]:
client = CAVEclient('minnie65_phase3_v1')

In [ ]:
ts = client.chunkedgraph.get_root_timestamps(root_id, latest=True)

In [ ]:
l2ids = client.chunkedgraph.get_roots(pc.nodes['pre_pt_supervoxel_id'], stop_layer=2, timestamp=ts[0])

In [ ]:
pc.add_label(l2ids, name='l2id_pre')

In [ ]:
syn_index_map = [l2_map[x] for x in l2ids if x in l2_map]

In [ ]:
nrn.add_point_annotations(
    'pre_syn',
    vertices=pc,
    linkage={'graph': np.array(syn_index_map)}
)

In [ ]:
nrn._morphsync

In [ ]:
nrn.annotations.names()

In [ ]:
nrn.

In [ ]:
nrn.add_skeleton(
    vertices=skel['vertices'],
    edges=skel['edges'],
    labels={'radius': skel['radius']},
)

In [ ]:
nrn._annotations.names()

In [ ]:
l2_df.reset_index(inplace=True)

In [ ]:
l2_df

In [ ]:
nrn.add_graph(
    vertices=
    spatial_columns=spatial_columns,
    edges=edges,
    properties=properties,
)

In [ ]:
l2_df['l2_id'].values

In [ ]:
%%timeit


In [ ]:
skel['mesh_to_skel_map']

In [ ]:
fastremap.renumber(
    l2_df.index.values,
)

In [ ]:
fastremap.remap(
    l2_graph,
)

In [ ]:
nrn.skeleton.labels

In [ ]:
pre_syn_df

In [ ]:
nrn.add_point_annotations(
    'pre_syn',
    vertices=pre_syn_df,
    spatial_columns=spatial_columns,
)

In [ ]:
nrn._morphsync.add_graph?

In [ ]:
pc.labels['size']

In [ ]:
mask = pc.get_label('size') > 4400

In [ ]:
pcm = pc._apply_mask(mask)

In [ ]:
pcm._links

In [ ]:
pc._morphsync.layers.loc[pc.name].layer

In [ ]:
layer = pc._morphsync.layers.loc[pc.name]

In [ ]:
layer = pc._morphsync.layers.loc[pc.name].layer

In [ ]:
pc._morphsync.

In [ ]:
pc._morphsync._layers.items()

In [ ]:
pc._morphsync._generate_new_morphology(pc.name, layer.vertices_index)

In [ ]:
pd.concat?

In [ ]:
pc._morphsync

In [ ]:
pc._morphsync.layers.loc['pre_syn'].layer

In [ ]:
knp.vstack(
    pre_syn_df[].values
)

In [ ]:
PointCloud(pre_syn_df)

In [ ]:
pre_syn_df.columns[~pre_syn_df.columns.isin(spatial_columns)]

In [ ]:
list(pd.DataFrame().columns)